# Retrieval Ablation: Embedders, Chunk Granularity, and Reranking

**Elie Bruno** &middot; SCIPER 355932 &middot; CS-552 Spring 2026 &middot; Team CiteRight

Compares three bi-encoders (OpenAI `text-embedding-3-small`, BGE-M3, E5-large) and ColBERTv2 late-interaction across coarse and fine chunks, with and without a cross-encoder reranker, scored on 50 gold queries with Precision@k, Recall@k, MRR, and Hit Rate.

---

This notebook is part of the **Faithful RAG** Open Project. It runs end-to-end inside the RCP pod launched by [`notebooks/submit.sh`](./submit.sh) with no manual setup: dependencies install in the bootstrap cell, the corpus downloads from [`citeright/corpus`](https://huggingface.co/datasets/citeright/corpus) into `/scratch`, and all generation runs locally on the 40 GB A100 by default. API-based comparisons are optional and skipped if no key is provided.

## Setup

Installs Python dependencies (idempotent on rerun), wires `sys.path` so we can `from evaluation.common import ...`, and resolves the corpus cache. On RCP this finds `/scratch/citeright_artifacts`; on a laptop it falls back to `data/s3_archive/` if you've cloned a local mirror, otherwise downloads from HuggingFace.

In [ ]:
%pip install --quiet huggingface_hub sentence-transformers faiss-cpu datasets transformers accelerate litellm tqdm pandas matplotlib

In [ ]:
import logging
import os
import sys
from pathlib import Path

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")

# Locate repo root (notebook lives in <repo>/notebooks/).
REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT.name == "notebooks" or not (REPO_ROOT / "pyproject.toml").exists():
    if REPO_ROOT.parent == REPO_ROOT:
        raise RuntimeError("Could not find repo root (no pyproject.toml above notebook).")
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# On RCP submit.sh sets these; default for laptop dev otherwise.
os.environ.setdefault("CITERIGHT_HF_REPO", "citeright/corpus")
if Path("/scratch").is_dir():
    os.environ.setdefault("CITERIGHT_DATA_DIR", "/scratch/citeright_artifacts")
    os.environ.setdefault("HF_HOME", "/scratch/hf_cache")

# Bootstrap imports only — pull additional helpers in your own cells:
#   from evaluation.common import generate, load_paper_markdown,
#       load_paper_chunks, load_paper_openalex, load_embedder, load_nli_classifier
from evaluation.common import (  # noqa: E402
    artifact_root,
    available_models,
    load_chunk_metadata,
    load_faiss_index,
    load_gold_qa,
)

ART = artifact_root()
print(f"Corpus root: {ART}")
print(f"Available models: {len(available_models())}")

### Optional: API keys

The notebook works with **only** local models. If you want the OpenAI / OpenRouter comparison arms (e.g. GPT-5, Claude, DeepSeek), paste the keys when prompted below. Leave blank to skip.

In [ ]:
# Optional: paid-API comparison arms.
# Skip these cells if you only want the local-model path (always works).
import getpass

for var in ("OPENAI_API_KEY", "OPENROUTER_API_KEY"):
    if var not in os.environ:
        val = getpass.getpass(f"{var} (leave blank to skip): ").strip()
        if val:
            os.environ[var] = val
print("Local models always available; API arms enabled iff keys above were provided.")

## Smoke test

Loads the FAISS index, the row-→-chunk metadata, and the gold evaluation set. If this cell errors, the rest of the notebook will not run — see `evaluation/common/data_loader.py` for the resolution rules.

In [ ]:
# Confirm the corpus loads end-to-end.
meta = load_chunk_metadata("coarse")
index = load_faiss_index("coarse")
gold = load_gold_qa()

print(f"Coarse FAISS rows: {index.ntotal}")
print(f"Coarse metadata entries: {len(meta)}")
print(f"Gold Q&A pairs: {len(gold)}")
assert index.ntotal == len(meta), "FAISS / metadata size mismatch"

# Show one chunk so we know the schema.
first = meta[0]
print(f"\nExample chunk:\n  paper_id: {first['paper_id']}")
print(f"  section : {first.get('section_hierarchy', [])}")
print(f"  text    : {first['text'][:200]}...")

## Build / load embeddings

Embed every chunk with each candidate model. The OpenAI vectors are pre-built (FAISS in `indexes/coarse.faiss`); BGE-M3 and E5-large run locally via `sentence-transformers`. Cache to `/scratch` so the second pass is free.

In [ ]:
# TODO: implement
raise NotImplementedError

## Run gold queries

For each (embedder × granularity × ±reranker) configuration, retrieve top-50 for every gold question. Save retrieved chunk IDs and scores to `evaluation/retrieval_eval/results/`.

In [ ]:
# TODO: implement
raise NotImplementedError

## Compute metrics

Use `evaluation.retrieval_eval.evaluate_retrieval.evaluate_retriever`. Tabulate Precision@{5,10,20}, Recall@{5,10,20}, MRR, Hit@k.

In [ ]:
# TODO: implement
raise NotImplementedError

## Plot ablation table & failure analysis

Bar plot per metric × granularity. Pull a handful of failure cases (gold not in top-20) and inspect what was retrieved instead.

In [ ]:
# TODO: implement
raise NotImplementedError

## Results & discussion

Summarise the headline numbers, ablations, and comparisons here. The 4-page report references plots and tables produced above.

## Reproduction

From a clean clone of the repo:

```bash
cd notebooks && ./submit.sh         # launch the RCP pod
runai port-forward <job> --port 8888:8888
# open http://localhost:8888 (token: cs552), then Run All on this notebook
```

All artefacts download automatically from [`citeright/corpus`](https://huggingface.co/datasets/citeright/corpus); no S3 or AWS access is required.